In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from typing import TypedDict
from dotenv import load_dotenv

C:\Users\kumanit\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Initialize Ollama model
model = ChatOllama(model="llama3.2:latest")

In [3]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    evaluate: int

In [4]:
def create_outline(state: BlogState) -> BlogState:

    #fetch title
    title = state['title']

    #call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    #update state
    state['outline'] = outline

    return state

In [5]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']

    outline = state['outline']

    prompt = f'write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [6]:
def evaluate_node(state: BlogState) -> BlogState:

    title = state['title']

    outline = state['outline']

    content = state['content']

    prompt = f"based on outline - {outline} try to rate my blog - {content}"

    evaluate = model.invoke(prompt).content

    state['evaluate'] = evaluate

    return state

In [7]:
graph = StateGraph(BlogState)

#nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate', evaluate_node)

#Edges

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate')
graph.add_edge('evaluate',END)

workflow = graph.compile()


In [ ]:
initial_state = {'title': 'Rise an AI in india'}

final_state = workflow.invoke(initial_state)

print(final_state)

In [ ]:
print(final_state['outline'])

In [ ]:
print(final_state['content'])

In [ ]:
print(final_state['evaluate'])